# Performer
You can __[download](https://arxiv.org/pdf/2009.14794)__ and read the paper. Also, there is a __[video](https://www.youtube.com/watch?v=xJrKIPwVwGM&t=76s)__ describes paper.

Here we want to implement Performer on __[LLAMA3](https://huggingface.co/meta-llama/Llama-3.2-1B)__ and compare it with Vanilla Transformer Attention mechanism.
## What expect?
Vanilla attention is O(L^2) and performer is O(L). There for, Performer should be faster and need less time and memmory.
## A brief look at the formulas
Vanilla transformer mechanism uses formula below to calculate:
$$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V$$
For Performer attention we use (FAVOR+):
$$\text{Attention}_{\text{FAVOR}+}(Q, K, V) = \frac{\Phi(Q) \left( \Phi(K)^\top V \right)}{\Phi(Q) \left( \Phi(K)^\top \mathbf{1} \right)}$$
## Implement Libraries that you need
Here we implemented Performer attention.

NOTE: Even we implemented vanilla attention, you can load LLAMA3 with vanilla attention. Just use line below:
```python
attn_implementation="eager"
```
### You need to run Language model downloaded from hugging face?
If You need some language model from hugging face, you need to implement huggingface_hub and use login function.

Use code below to implement it:
```python
from huggingface_hub import login
login()
```

You need __[Access Token](https://huggingface.co/docs/hub/en/security-tokens)__ for that. You can __[crete your access token](https://huggingface.co/settings/tokens)__ by your own.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
import math
import bitsandbytes as bnb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb
from huggingface_hub import login
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm

# Do you want to use huggingface models?
First login to huggingface

In [ ]:
login()

## Load your model
You can load any model that you want. Just use huggingface link pass it as funtion input. As default model, we set it LLAMAA-3.2-1B and quantized it to 4bit.
### Want to use small models?
Take it easy. Just disable quantization using quantize parameter. Make it false
### SMALL TIP
If you use small models, you may not get your prefered output. Use models that can handel inputs with more than 4096 inputs (The reason is mentioned in next cells).

In [ ]:
def generate_model(model_id="meta-llama/Llama-3.2-1B", quantize=True, device="cuda"):

    if quantize:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        bnb_config = None
        
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).to(device)
    model = prepare_model_for_kbit_training(model)
    return model, tokenizer

model, tokenizer = generate_model()

# Implement Performer function using this class
First we need to implement the matrixes with the sizes. As we use 4-bit quantized model, here we need to implement for this model too. So, there is an opthion `quantized` that you can set it True to implement.

$$\phi(x) = \frac{h(x)}{\sqrt{m}} \exp(W \cdot x)$$ $$h(x) = \exp\left(-\frac{\|x\|^2}{2}\right)$$

## Calculate output
To calculate output
### Implementation Logic (Causal FAVOR+)
The code implements the causal mechanism by separating the numerator (values) and the denominator (normalization) to achieve linear complexity $O(L \cdot d \cdot m)$:

$$\text{Output}_i = \frac{\phi(q_i)^\top \sum_{j=1}^i (\phi(k_j) \otimes v_j)}{\phi(q_i)^\top \sum_{j=1}^i \phi(k_j) + \epsilon}$$

#### Correspondence with Code:
* **Numerator:** `(q_prime * torch.cumsum(k_prime @ v, dim=2)).sum(dim=3)`
* **Denominator:** `(q_prime * torch.cumsum(k_prime, dim=2)).sum(dim=3)`
* **$\epsilon$:** `1e-6` (added for numerical stability).

In [ ]:
class PerformerSelfAttention(nn.Module):
    def __init__(self, config, nb_features=256, quantized=True):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_groups = self.num_heads // self.num_kv_heads
        self.nb_features = nb_features

        if quantized:
            # 4-bit quantized parameter set
            self.q_proj = bnb.nn.Linear4bit(
                self.hidden_size, self.num_heads * self.head_dim, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            self.k_proj = bnb.nn.Linear4bit(
                self.hidden_size, self.num_kv_heads * self.head_dim, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            self.v_proj = bnb.nn.Linear4bit(
                self.hidden_size, self.num_kv_heads * self.head_dim, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            self.o_proj = bnb.nn.Linear4bit(
                self.num_heads * self.head_dim, self.hidden_size, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
        else:
            # Standard parameter set
            self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
            self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

        projection_matrix = torch.randn(self.nb_features, self.head_dim)
        q, _ = torch.qr(projection_matrix.T) # Orthogonalize for better accuracy
        self.register_buffer("projection_matrix", q.T)

    def repeat_kv(self, hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
        batch, num_key_value_heads, slen, head_dim = hidden_states.shape
        if n_rep == 1:
            return hidden_states
        hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
        return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

    def feature_map(self, x):
        # Project x onto the random features
        x_proj = x @ self.projection_matrix.T

        # Calculate norm term for the softmax approximation: ||x||^2 / 2
        x_norm = torch.sum(x**2, dim=-1, keepdim=True) / 2.0

        # The positive feature map (FAVOR+ SMU kernel)
        phi = torch.exp(x_proj - x_norm - x_proj.max(dim=-1, keepdim=True)[0]) # Max-subtraction for stability
        return phi / math.sqrt(self.nb_features)

    def forward(self, hidden_states, attention_mask=None, position_ids=None, **kwargs):
        Batch, T, C = hidden_states.size()

        q = self.q_proj(hidden_states).view(Batch, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(Batch, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(Batch, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        k = self.repeat_kv(k, self.num_groups)
        v = self.repeat_kv(v, self.num_groups)

        q_prime = self.feature_map(q) # (B, H, T, M)
        k_prime = self.feature_map(k) # (B, H, T, M)

        batch, h, t, m = k_prime.shape # (B, H, T, M, D)
        d = v.shape[-1]

        k_v_outer = k_prime.view(batch, h, t, m, 1) * v.view(batch, h, t, 1, d)
        batch, h, t, m = q_prime.shape

        kv_cumsum = torch.cumsum(k_v_outer, dim=2)
        out_num = (q_prime.view(batch, h, t, m, 1) * kv_cumsum).sum(dim=3)
        k_cumsum = torch.cumsum(k_prime, dim=2) # (B, H, T, M)
        out_den = (q_prime * k_cumsum).sum(dim=3)
        out_den = out_den.unsqueeze(-1) + 1e-6 # Add epsilon for stability

        # 4. Normalize and Project
        output = out_num / out_den
        output = output.transpose(1, 2).contiguous().view(Batch, T, C)

        return self.o_proj(output), None

In [ ]:
for layer in model.model.layers:
    layer.self_attn = PerformerSelfAttention(layer.self_attn.config).to("cuda")